## Document RAG analysis with Closed Source LLMs

In [1]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
load_dotenv()


file_path = './files/ipaybtc-faq.txt'
# Load Document
with open(file_path) as f:
    doc = f.read()
    
# Split Document
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 150,
    chunk_overlap=20,
    length_function=len
)    

# Create chunks
chunks = text_splitter.split_text(doc)

print(f"Chunks: {len(chunks)}")

for i, chunk in enumerate(chunks):
    print(f"Chunk {i} - {chunk} \n")

/Users/inventmbp/Documents/projects/ml-ai-projects/rag-projects/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunks: 131
Chunk 0 - Our Vision 

Chunk 1 - ipayBTC has a bold vision of building a world where satoshi is the dominant unit of account, a world where the majority of people earn, save and 

Chunk 2 - earn, save and spend their value in satoshi. 

Chunk 3 - Our Mission
ipayBTC aims to incorporate satoshi payments into all facets of our everyday life. 

Chunk 4 - Cross-border Payments
With ipayBTC, users can receive instant and inexpensive cross-border payments using the Lightning Network. 

Chunk 5 - Micro Payments
ipayBTC supports the Lightning Network, allowing small businesses and freelancers to handle small and micro-transactions efficiently. 

Chunk 6 - Financial Inclusion
ipayBTC addresses financial inclusion by educating users through various content formats on every platform it is present on. 

Chunk 7 - Which ID verification type should I use? 

Chunk 8 - To ensure security and smooth customer experience, we require customers to verify their identity. For ID verification, you

In [2]:
# Embeddings
from sentence_transformers import SentenceTransformer

# Load embedding model
sentenceModel = SentenceTransformer('BAAI/bge-small-en-v1.5')

chunk_emdeddings = sentenceModel.encode(chunks)

print(f"Shape of embeddings: {chunk_emdeddings.shape}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2644.76it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape of embeddings: (131, 384)


In [3]:
# Vector store
import faiss
import numpy as np

# Get dimension
d = chunk_emdeddings.shape[1]

# Create a FAISS index
index = faiss.IndexFlatL2(d)

# Add chunk embeddings to the index
index.add(np.array(chunk_emdeddings).astype('float32'))

print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 131 vectors.


In [4]:
# Open AI LLM
import os
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY) 

In [5]:
def get_context(prompt):
    query_embedding = sentenceModel.encode([prompt]).astype('float32')

    # Search the FAISS index for the top k
    k = 5
    distances, indices = index.search(query_embedding, k)

    # Get the text chunks from chunks list
    retrieved_chunks = [chunks[i] for i in indices[0]]
    
    context = "\n\n".join(retrieved_chunks)
    
    return context

In [18]:

def generate_answer_openai(prompt):
    try:
        
        context = get_context(prompt)

        print(f"Question: {prompt} \n\n")
        print(f"Context: {context}")
    
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system", 
                    "content": """
                        You are a friendly customer support representative.

                        Use ONLY the provided context.
                        Do NOT add new information.
                        Do NOT infer or assume anything not explicitly stated.

                        If the context does not contain the answer, say:
                        "Sorry, I don't have that information."

                        Be polite and concise (1-2 sentences).
                    """
                 },
                {"role": "user", "content": f"Context: {context}\n\n Customer Issue: {prompt}"}
            ],
            max_tokens=50,
        )
        
    
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error initializing OpenAI client: {e}")   

In [7]:
# Store past 3 questions
# Rewrite/summarize past questions to improve retrieval
# Detect follow up questions, use history of previous questions and answers.

In [17]:
prompt = "How do I receive Bitcoin?"
answer = generate_answer_openai(prompt)
print(f"Answer: {answer}")

Question: How do I receive Bitcoin? 


Context: How to receive Bitcoin (on chain)?
Go to your iPayBTC wallet and click "Receive Bitcoin."
Select On-Chain Network.

How to receive Bitcoin (lightning network)?
Go to your iPayBTC wallet and click "Receive Bitcoin."
Select "Lightning Network."

Once the sender completes the payment, you'll receive Bitcoin instantly!

Input the amount you want to send.
Review transaction details and confirm.
The transaction will be processed and recorded on the Bitcoin blockchain

A partner sent me some Bitcoin and I have not received it in my iPayBTC wallet
['How can i receive my bitcoin', 'Okay', 'How do I reset my password?', 'How do I receive Bitcoin?']
Answer: To receive Bitcoin, go to your iPayBTC wallet and click "Receive Bitcoin." Select either the On-Chain Network or the Lightning Network, depending on how the sender is sending it.
